# Manual Training Loop

Goal: understand what happens during neural-network training by manually training a tiny model.

We will learn a simple relationship:

y = 3x + 2

and watch PyTorch adjust a weight and bias using gradients.

In [73]:
# simple training data

import torch

x = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0],
    [5.0]
])

y = 3 * x + 2  #y = 3x + 2, behind the scenes we know weights = 3 and bias = 2

print("x:\n", x)
print("\ny:\n", y)

x:
 tensor([[1.],
        [2.],
        [3.],
        [4.],
        [5.]])

y:
 tensor([[ 5.],
        [ 8.],
        [11.],
        [14.],
        [17.]])


In [74]:
# intentionally incorrect starting parameters for weights and bias

w = torch.tensor([[0.5]], requires_grad=True)  # requires_grad=True tells pytorch to track how calculations depend on w and b so that we can retrieve their gradients
b = torch.tensor([0.0], requires_grad=True) # these parameters w and b are the trainable parameters of our model, and we will update them during training to minimize the loss function

print("Starting weight:", w.item())
print("Starting bias:", b.item())

Starting weight: 0.5
Starting bias: 0.0


In [75]:
# manual predictions

predictions = x @ w + b  # yhat = x @ w + b, where @ is the matrix multiplication operator in pytorch

print(predictions)

tensor([[0.5000],
        [1.0000],
        [1.5000],
        [2.0000],
        [2.5000]], grad_fn=<AddBackward0>)


In [76]:
# loss calculation

loss = ((predictions - y) ** 2).mean()

print("Loss:", loss.item())

Loss: 102.75


In [77]:
loss.backward() 

# loss function moves backwards through the chain
# gradient values tell the direction and magnitude w and b should move to in order to reduce loss

print("Gradient for w:", w.grad)
print("Gradient for b:", b.grad)

Gradient for w: tensor([[-67.]])
Gradient for b: tensor([-19.])


In [78]:
# manual gradient descent update (once)

learning_rate = 0.01

with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

print("Updated weight:", w.item())
print("Updated bias:", b.item())

Updated weight: 1.1699999570846558
Updated bias: 0.1899999976158142


In [79]:
# clear gradients (pytorch accumulates gradients)
# if we dont clear gradients the next backward() will add new gradients to old ones
# real pytorch training loops use optimizer.zero_grad() for this

w.grad.zero_()
b.grad.zero_()

tensor([0.])

### Full Manual Training Loop

In [80]:
learning_rate = 0.01
epochs = 2000

for epoch in range(epochs):

    # Forward pass
    predictions = x @ w + b # use current w and b to make initial predictions

    # Calculate loss
    loss = ((predictions - y) ** 2).mean() # measure how wrong the initial predictions are

    # Backward pass
    loss.backward() # calculate gradients

    # Update parameters
    with torch.no_grad():
        w -= learning_rate * w.grad # adjust weight based on gradients
        b -= learning_rate * b.grad # adjust bias based on gradients

    # Clear old gradients
    w.grad.zero_() # clear weight gradient
    b.grad.zero_() # clear bias gradient

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Loss: {loss.item():.8f} | "
            f"w: {w.item():.4f} | "
            f"b: {b.item():.4f}"
        )

Epoch   0 | Loss: 59.98780060 | w: 1.6812 | b: 0.3360
Epoch  10 | Loss: 0.51889551 | w: 3.2119 | b: 0.8032
Epoch  20 | Loss: 0.23405032 | w: 3.3047 | b: 0.8708
Epoch  30 | Loss: 0.21758945 | w: 3.3013 | b: 0.9102
Epoch  40 | Loss: 0.20333405 | w: 3.2917 | b: 0.9466
Epoch  50 | Loss: 0.19001769 | w: 3.2820 | b: 0.9817
Epoch  60 | Loss: 0.17757311 | w: 3.2727 | b: 1.0156
Epoch  70 | Loss: 0.16594392 | w: 3.2636 | b: 1.0484
Epoch  80 | Loss: 0.15507606 | w: 3.2548 | b: 1.0801
Epoch  90 | Loss: 0.14491998 | w: 3.2463 | b: 1.1107
Epoch 100 | Loss: 0.13542905 | w: 3.2381 | b: 1.1403
Epoch 110 | Loss: 0.12655956 | w: 3.2302 | b: 1.1690
Epoch 120 | Loss: 0.11827103 | w: 3.2225 | b: 1.1966
Epoch 130 | Loss: 0.11052550 | w: 3.2151 | b: 1.2234
Epoch 140 | Loss: 0.10328700 | w: 3.2079 | b: 1.2492
Epoch 150 | Loss: 0.09652267 | w: 3.2010 | b: 1.2742
Epoch 160 | Loss: 0.09020139 | w: 3.1943 | b: 1.2984
Epoch 170 | Loss: 0.08429409 | w: 3.1879 | b: 1.3218
Epoch 180 | Loss: 0.07877354 | w: 3.1816 | b:

# Implementing a PyTorch Optimizer

replacing manual gradient-descent with "torch.optim.SGD"

In [81]:
# reset parameters

w = torch.tensor([[0.5]], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

print("Starting weight:", w.item())
print("Starting bias:", b.item())

Starting weight: 0.5
Starting bias: 0.0


In [83]:
learning_rate = 0.01

# pytorch optimizer function
# assigning parameters w and b to the function for updating
optimizer = torch.optim.SGD( 
    [w, b],
    lr=learning_rate
)